# Week 4 · Tutorial 3 — Ratios and aggregates, periodicity, and data leakage

**Covers videos 4.8 – 4.10:** variables built from other variables · putting the clock back on a circle · where a good result comes from a broken experiment.

**Dataset:** `smart_meters_hourly.csv` — hourly electricity use of 40 non-residential and apartment-building meters, 9 January – 9 March 2023.
**One row is one meter-hour.** Columns: `meter_id`, `timestamp`, `kwh`, `floor_area_m2`, `building_type` (`office`, `school`, `shop`, `flat block`), `temperature_c`.

No predictive model is used in this notebook. Instead, we construct features and *measure* how leakage inflates results using simple baselines.

### How to work with this notebook (no programming experience needed)

* A notebook is a list of **cells**. Click a code cell and press **Shift + Enter** to run it; the result appears below it. Run the cells **from top to bottom**.
* Cells that start with `import` or read a file (`pd.read_csv(...)`) just load tools and data. You do not need to understand every line.
* In every **✏️ Your turn** block you make one small, exactly described change (a number, a word, or a line to paste in), then run the cell again.
  The box **What to change** shows the exact text to type. Lines marked `# TODO` are the ones to edit.
* Write down the numbers that appear — the quiz asks for them. If you break something, re-run the cells above it, or use *Kernel → Restart & Run All*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
sm = pd.read_csv("data/smart_meters_hourly.csv", parse_dates=["timestamp"]).sort_values(["meter_id", "timestamp"]).reset_index(drop=True)
print(sm.shape, "|", sm["meter_id"].nunique(), "meters |", sm["timestamp"].min(), "to", sm["timestamp"].max())
sm.head()

## 1. Ratios and aggregates

### Energy per square metre

A shop of 1 500 m² uses more than a flat block of 400 m² just because it is bigger. The **intensity** `kWh / m²` compares buildings of different size.

### ✏️ Your turn 1

Create the intensity column `wh_per_m2` = `kwh` / `floor_area_m2` × 1000 (watt-hours per square metre per hour) and compute its mean per `building_type`. Fill in the `TODO`s.

**What to change** — edit the cell exactly like this, then run it:

Change `sm["wh_per_m2"] = np.nan` into `sm["wh_per_m2"] = sm["kwh"] / sm["floor_area_m2"] * 1000`

In [ ]:
sm["wh_per_m2"] = np.nan        # TODO

by_type = sm.groupby("building_type")["wh_per_m2"].mean().round(2)
print(by_type)
if by_type.notna().any():
    print("highest intensity:", by_type.idxmax(), "| lowest:", by_type.idxmin())
    print("highest / lowest:", round(by_type.max() / by_type.min(), 2))

### How does a meter compare to its own group?

Dividing a meter's intensity by the **mean intensity of its building type at the same hour** removes the common daily rhythm and the weather,
and leaves the meter's own behaviour (1 = typical for its type).

### ✏️ Your turn 2

Fill in the `TODO`: `group_mean` = mean of `wh_per_m2` over all meters of the same `building_type` at the same `timestamp` (`groupby(["building_type", "timestamp"])["wh_per_m2"].transform("mean")`),
then `relative = wh_per_m2 / group_mean`. The cell reports the meter with the highest mean `relative` value and that value (3 decimals), and the meter with the lowest.

**What to change** — edit the cell exactly like this, then run it:

Change `sm["group_mean"] = np.nan` into `sm["group_mean"] = sm.groupby(["building_type", "timestamp"])["wh_per_m2"].transform("mean")`

In [ ]:
sm["group_mean"] = np.nan        # TODO
sm["relative"] = sm["wh_per_m2"] / sm["group_mean"]

rel = sm.groupby("meter_id")["relative"].mean()
if rel.notna().any():
    print("highest:", rel.idxmax(), round(rel.max(), 3), "| type:", sm.loc[sm["meter_id"] == rel.idxmax(), "building_type"].iloc[0])
    print("lowest :", rel.idxmin(), round(rel.min(), 3))

### Buildings have thermal mass

A building does not react to the temperature *now* but to the temperature of the last hours or days. A **rolling mean** of the temperature is a better feature than the temperature itself.
We work with daily values: total energy of all meters per day vs. the rolling mean of the daily mean temperature over a window of `w` days.

In [ ]:
hourly = sm.groupby("timestamp").agg(total_kwh=("kwh", "sum"), temp=("temperature_c", "first"))
daily = hourly.resample("D").agg({"total_kwh": "sum", "temp": "mean"})
print(daily.shape[0], "days")
print("correlation of daily load with today's mean temperature:", round(daily["total_kwh"].corr(daily["temp"]), 3))

### ✏️ Your turn 3

Which window is best? Fill in the `TODO`: for each window `w` in `[1, 2, 3, 4, 5, 7]` (days) compute the correlation between `daily["total_kwh"]` and
`daily["temp"].rolling(w).mean()` (pandas ignores the first `w − 1` days automatically) and store it in `corr_by_w`.
The cell reports the best window (largest absolute correlation) and its correlation (3 decimals), and the correlation for `w = 7`.

**What to change** — edit the cell exactly like this, then run it:

Change `pass` into `corr_by_w[w] = daily["total_kwh"].corr(daily["temp"].rolling(w).mean())`

In [ ]:
corr_by_w = {}
for w in [1, 2, 3, 4, 5, 7]:
    pass      # TODO

if corr_by_w:
    best_w = max(corr_by_w, key=lambda k: abs(corr_by_w[k]))
    print("best window:", best_w, "days | correlation:", round(corr_by_w[best_w], 3))
    print("window 7:", round(corr_by_w[7], 3))

### Lagged consumption

The consumption one hour ago or 24 hours ago is a feature known at prediction time. The consumption **one hour ahead** is not — it is the thing we want to know.
`groupby("meter_id")[...].shift(k)` moves values `k` rows *later* in time (past values) — **per meter**, so that one meter's data does not leak into the next.

### ✏️ Your turn 4

Fill in the three `TODO`s with `groupby("meter_id")["wh_per_m2"].shift(...)`: `lag_1` = shift(1), `lag_24` = shift(24), and `lead_1` = shift(−1) (a **future** value).
The cell prints the correlation of each with `wh_per_m2` (3 decimals).

Then answer: which of the three can be used as an input when predicting the next hour?

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
sm["lag_1"] = np.nan
sm["lag_24"] = np.nan
sm["lead_1"] = np.nan
```
by
```python
sm["lag_1"] = g.shift(1)
sm["lag_24"] = g.shift(24)
sm["lead_1"] = g.shift(-1)
```

In [ ]:
g = sm.groupby("meter_id")["wh_per_m2"]
sm["lag_1"] = np.nan        # TODO
sm["lag_24"] = np.nan       # TODO
sm["lead_1"] = np.nan       # TODO

print(sm[["lag_1", "lag_24", "lead_1"]].corrwith(sm["wh_per_m2"]).round(3))
print("rows without lag_24 (first day of each meter):", int(sm["lag_24"].isna().sum()))

## 2. Periodicity: putting the clock back on a circle

Hour 23 and hour 0 are neighbours, but as numbers they are 23 apart. Encode the hour as a point on a circle:
`sin(2π · hour / 24)` and `cos(2π · hour / 24)`.

### ✏️ Your turn 5

Fill in the `TODO`s for the office and school meters (`work`): `hour_sin`, `hour_cos` (using `np.sin`/`np.cos`) and report the correlation of `wh_per_m2` with `hour`,
`hour_sin` and `hour_cos` (3 decimals). Also compute `dist_23_0`, the Euclidean distance between hour 23 and hour 0 on the circle (3 decimals), and `dist_11_12` for hours 11 and 12.

**What to change** — edit the cell exactly like this, then run it:

1. Replace
```python
work["hour_sin"] = np.nan
work["hour_cos"] = np.nan
```
by
```python
work["hour_sin"] = np.sin(2 * np.pi * work["hour"] / 24)
work["hour_cos"] = np.cos(2 * np.pi * work["hour"] / 24)
```

2. Change `return np.array([np.nan, np.nan])` into `return np.array([np.sin(2 * np.pi * h / 24), np.cos(2 * np.pi * h / 24)])`

In [ ]:
work = sm[sm["building_type"].isin(["office", "school"])].copy()
work["hour"] = work["timestamp"].dt.hour

work["hour_sin"] = np.nan      # TODO
work["hour_cos"] = np.nan      # TODO

def circle(h):
    return np.array([np.nan, np.nan])      # TODO: (sin, cos) of hour h

dist_23_0 = np.linalg.norm(circle(23) - circle(0))
dist_11_12 = np.linalg.norm(circle(11) - circle(12))

print(work[["hour", "hour_sin", "hour_cos"]].corrwith(work["wh_per_m2"]).round(3))
print("distance 23 -> 0 on the circle :", round(dist_23_0, 3), "(as numbers: 23)")
print("distance 11 -> 12 on the circle:", round(dist_11_12, 3), "(as numbers: 1)")

### Sine waves cannot do holidays

A winter break is not periodic. Add it as a **flag**. In this region the schools were closed Monday 20 – Friday 24 February 2023.

### ✏️ Your turn 6

Fill in the `TODO`: `school_closed` = 1 for timestamps from `2023-02-20` to `2023-02-24 23:59` (use `between`). Then compare the mean `wh_per_m2` of the **school** meters on
weekdays (Monday – Friday) with the flag set vs. not set, and report the ratio (closed / open, 3 decimals).

**What to change** — edit the cell exactly like this, then run it:

Change `sc["school_closed"] = 0` into `sc["school_closed"] = sc["timestamp"].between("2023-02-20", "2023-02-24 23:59").astype(int)`

In [ ]:
sc = sm[sm["building_type"] == "school"].copy()
sc["weekday"] = sc["timestamp"].dt.dayofweek
sc["school_closed"] = 0      # TODO: 1 between 2023-02-20 and 2023-02-24 23:59

wd = sc[sc["weekday"] < 5]
means = wd.groupby("school_closed")["wh_per_m2"].mean()
print(means.round(2))
if 1 in means.index:
    print("closed / open:", round(means[1] / means[0], 3))
    print("closed hours (all school meters):", int(sc["school_closed"].sum()))

## 3. Data leakage: a good result from a broken experiment

**Before you believe a result:** was everything fitted on the training data only? Does the split match the deployment? Is every input known at prediction time? Were duplicates removed before splitting?

### Type 1: preprocessing

In [ ]:
meter = sm[sm["meter_id"] == "M00"].reset_index(drop=True)
n_tr = int(0.7 * len(meter))
tr_part, te_part = meter.iloc[:n_tr], meter.iloc[n_tr:]
print(meter["building_type"].iloc[0], "| hours:", len(meter), "| train:", len(tr_part), "| test:", len(te_part))

### ✏️ Your turn 7

Standardising a column with statistics from the **whole** data set lets information from the test period leak into the training pipeline.

Change `fit_on` from `meter` (all hours) to `tr_part` (training hours only). The cell standardises `kwh` with the mean and standard deviation of `fit_on` and reports the mean and standard deviation of the standardised **test** hours (3 decimals) — and the mean and standard deviation that were used.

**What to change** — edit the cell exactly like this, then run it:

Change `fit_on = meter` into `fit_on = tr_part`

In [ ]:
fit_on = meter           # TODO: tr_part

mu, sd = fit_on["kwh"].mean(), fit_on["kwh"].std()
test_scaled = (te_part["kwh"] - mu) / sd

print("mean and std used for the scaling:", round(mu, 3), round(sd, 3))
print("standardised test hours: mean", round(test_scaled.mean(), 3), "| std", round(test_scaled.std(), 3))

### Type 2: the split

We predict the consumption of meter M00 for each test hour with a very simple **interpolation baseline**: linearly interpolate between the nearest *training* hours on both sides.
If test hours are scattered randomly, neighbours are always close and the baseline looks great. If the test period is the future, there are no neighbours on the right.

### ✏️ Your turn 8

Compare a **random** split with a **chronological** split (both: 30 % test).

Change `split = "random"` to `split = "chronological"` and run again. The cell reports the mean absolute error (MAE, 3 decimals; unit kWh) of the interpolation baseline on the test hours. Note both values.

In [ ]:
split = "random"        # TODO: "chronological"

rng = np.random.default_rng(0)
idx = np.arange(len(meter))
if split == "random":
    test_mask = np.zeros(len(meter), dtype=bool)
    test_mask[rng.choice(len(meter), size=int(0.3 * len(meter)), replace=False)] = True
else:
    test_mask = idx >= int(0.7 * len(meter))

pred = np.interp(idx[test_mask], idx[~test_mask], meter.loc[~test_mask, "kwh"])
mae = np.mean(np.abs(pred - meter.loc[test_mask, "kwh"]))
print("split:", split, "| test hours:", int(test_mask.sum()))
print("MAE of the interpolation baseline:", round(float(mae), 3))

### Duplicates and target encoding

**Duplicates** that end up on both sides of a split are the simplest leak. And **target encoding** computed on all the data (including the row itself) leaks the answer directly:
here we give every hour of meter M00 a *random* label (one of 400 meaningless IDs) and encode the label by the mean consumption of rows with the same label.
A meaningless label should have a correlation of about 0 with the consumption.

### ✏️ Your turn 9

Change `fit_rows` from `meter` (all rows) to `tr_part` (training rows only) and compare. The cell computes the mean consumption per label from `fit_rows`, encodes **test** rows with it (unseen labels get the overall mean of `fit_rows`)
and reports the correlation between the encoded label and the true `kwh` on the test hours (3 decimals).
When `fit_rows` is `meter`, the encoded value of a test row contains that row's own consumption.

In [ ]:
rng = np.random.default_rng(5)
labels = pd.Series(rng.integers(0, 400, len(meter)), index=meter.index)

fit_rows = meter            # TODO: tr_part

label_mean = meter["kwh"].groupby(labels).mean() if fit_rows is meter else tr_part["kwh"].groupby(labels.loc[tr_part.index]).mean()
encoded_test = labels.loc[te_part.index].map(label_mean).fillna(fit_rows["kwh"].mean())

print("labels with an encoding:", len(label_mean))
print("correlation of the encoded label with kwh in the test hours:", round(encoded_test.corr(te_part["kwh"]), 3))

### ✏️ Your turn 10

Duplicates. A faulty export wrote every hour of meter M00 twice (`dup = pd.concat([meter, meter])`). The cell makes a random 30 % test split of the rows and reports the share of
test rows whose timestamp also occurs in the training rows.

Fill in the `TODO` to remove the duplicates *before* splitting (`dup.drop_duplicates()`), and run again.

In [ ]:
dup = pd.concat([meter, meter], ignore_index=True)
# TODO: remove duplicates before the split

rng = np.random.default_rng(1)
is_test = np.zeros(len(dup), dtype=bool)
is_test[rng.choice(len(dup), size=int(0.3 * len(dup)), replace=False)] = True
train_ts = set(dup.loc[~is_test, "timestamp"])

print("rows:", len(dup), "| test rows:", int(is_test.sum()))
print("share of test rows whose timestamp is in the training rows:", round(dup.loc[is_test, "timestamp"].isin(train_ts).mean(), 3))

## Summary

* Ratios (per area), group-relative values, rolling means (thermal mass) and lags make the *shape* of the question easier — they add no information. Always check that a feature is **known at prediction time** (lags: yes; leads: no).
* Put circular variables on a circle (sin/cos) and add flags for events that are not periodic.
* Leakage inflates results: fit every transformation on training data only, split the way production will see new data, and remove duplicates **before** splitting.